# Tutorial - Base

First, let's verify that `Boltz-tools` is installed correctly by running:

In [ ]:
!boltz-lab --version

If the installation was successful, this command will display the installed version number of `Boltz-tools`.

If you encounter any errors, refer to the installation guide in the `README.md` file to troubleshoot and complete the setup.

> **Note:**  
> - When running these commands in a regular command-line interface (CLI), remove the `!` at the beginning of the command.
> - If you're copying commands from a Jupyter Notebook that use backslashes (`\`) to split long commands across multiple lines, remove the backslashes and write the command on a single line in the CLI.

In [ ]:
!boltz-lab predict --help

In [49]:
!boltz-lab predict \
        -w ./tmp/tutorial_base/ \
        -s ../examples/system.yaml \
        -b ../examples/options.yaml \
        --repeats 2 \
        --seed 42 \
        --conformers 3D \
        --log_name log_predict \
        --debug

/home/remco/.conda/envs/boltz-tools/lib/python3.12/site-packages/MDAnalysis/topology/tables.py:52: DeprecationWarning: Deprecated in version 2.8.0
MDAnalysis.topology.tables has been moved to MDAnalysis.guesser.tables. This import point will be removed in MDAnalysis version 3.0.0
  warnings.warn(wmsg, category=DeprecationWarning)
2026-01-12 15:01:19,212 | INFO     | cli.py:147 | setup | Logger initialized. Log file: tmp/tutorial_base/log_predict.log, debug=True
2026-01-12 15:01:19,213 | INFO     | cli.py:165 | main | Starting Boltz-lab prediction pipeline.
2026-01-12 15:01:19,213 | DEBUG    | predict.py:65 | __init__ | Initializing Predict with parameters: {'wrk_dir': PosixPath('tmp/tutorial_base'), 'system_path': PosixPath('../examples/system.yaml'), 'options_path': PosixPath('../examples/options.yaml'), 'repeats': 2, 'seed': 42, 'conformers': '3D', 'sdf_file': None}
2026-01-12 15:01:19,213 | INFO     | read.py:124 | read_yaml | ../examples/options.yaml loaded successfully. Contents:


In [50]:
import os
import pickle
import pandas as pd

# Path to folder containing IFP pickle files
folder_path = "./tmp/tutorial_base/results/ifp/prolif"
pickle_files = [f for f in os.listdir(folder_path) if f.endswith(".pkl")]

dfs = []

for file_name in pickle_files:
    file_path = os.path.join(folder_path, file_name)
    
    # Load the IFP
    with open(file_path, "rb") as f:
        loaded_ifp = pickle.load(f)
    
    # Dictionary to store binary matrix for this ligand/file
    binary_dict = {}

    for (lig_res, prot_res), interactions in loaded_ifp.items():
        for inter_type, inter_data_tuple in interactions.items():
            for inter_data in inter_data_tuple:
                prot_indices = inter_data['parent_indices']['protein']
                for prot_idx in prot_indices:
                    # Make column name: RESNAME + RESNUM + CHAIN + interaction
                    col_name = f"{prot_res.name}{prot_res.number}_{inter_type}"
                    binary_dict[col_name] = True

    # Convert to DataFrame (row = ligand/file)
    df = pd.DataFrame(binary_dict, index=[os.path.splitext(file_name)[0]])
    dfs.append(df)

# Merge all dataframes (rows = ligands/files, columns = residue + interaction type)
combined_df = pd.concat(dfs, axis=0).fillna(False).astype(bool)

# Optional: sort columns by residue number then interaction type
def sort_key(col_name):
    import re
    # Extract residue number
    match = re.match(r"[A-Z]{3}(\d+)([A-Z])?_(.*)", col_name)
    if match:
        resnum = int(match.group(1))
        inter_type = match.group(3)
        return resnum, inter_type
    return 0, col_name

combined_df = combined_df.reindex(sorted(combined_df.columns, key=sort_key), axis=1)
combined_df = combined_df.sort_index(axis=0)
combined_df

/tmp/ipykernel_4155386/1609252236.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  combined_df = pd.concat(dfs, axis=0).fillna(False).astype(bool)


,LEU7_Hydrophobic,LEU7_VdWContact,GLY8_VdWContact,SER9_VdWContact,PHE12_Hydrophobic,PHE12_VdWContact,VAL15_Hydrophobic,VAL15_VdWContact,ALA32_Hydrophobic,ALA32_VdWContact,...,ARG130_HBDonor,ARG130_VdWContact,ASN131_HBDonor,ASN131_VdWContact,LEU133_Hydrophobic,LEU133_VdWContact,THR143_Hydrophobic,THR143_VdWContact,ASP144_Hydrophobic,ASP144_VdWContact
1_system_model_0_ligand_chain_B_ifp,False,False,False,False,False,False,False,False,True,True,...,False,True,False,True,True,True,True,False,True,True
1_system_model_0_ligand_chain_C_ifp,True,True,True,True,True,True,True,True,True,False,...,False,False,False,False,False,False,False,False,False,False
1_system_model_1_ligand_chain_B_ifp,False,False,False,False,False,False,True,True,True,False,...,True,True,True,True,True,False,True,False,True,True
1_system_model_1_ligand_chain_C_ifp,True,True,False,False,False,False,False,False,True,True,...,False,False,False,False,True,True,False,True,False,False
2_system_model_0_ligand_chain_B_ifp,False,False,False,False,False,False,True,False,True,False,...,False,True,False,False,True,True,True,True,False,False
2_system_model_0_ligand_chain_C_ifp,True,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2_system_model_1_ligand_chain_B_ifp,True,True,False,False,False,False,True,True,True,False,...,False,False,False,False,True,True,False,False,False,False
2_system_model_1_ligand_chain_C_ifp,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
